Task 3 — Correlation Between News Sentiment and Stock Movement
Predicting Price Moves with News Sentiment

This notebook performs sentiment analysis and correlation analysis between financial news headlines and stock price movements for the following stocks:

AAPL
AMZN
GOOG
META
NVDA

The objective is to determine whether news sentiment has a measurable relationship with daily stock returns using statistical correlation analysis.

1. Import Libraries

In [5]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sentiment Analysis
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# Statistical analysis
from scipy.stats import pearsonr

# Date handling
from pandas.tseries.offsets import BDay

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Download VADER lexicon
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...


True

2. Load Datasets

In [7]:
# Load financial news dataset
news_df = pd.read_csv("../data/raw/raw_analyst_ratings.csv")

# Display first rows
news_df.head()

,Unnamed: 0,headline,url,publisher,date,stock
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A
2,2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A
3,3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A
4,4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A


In [8]:
# Load stock datasets

aapl_df = pd.read_csv("../data/Data/AAPL.csv")
amzn_df = pd.read_csv("../data/Data/AMZN.csv")
goog_df = pd.read_csv("../data/Data/GOOG.csv")
meta_df = pd.read_csv("../data/Data/META.csv")
nvda_df = pd.read_csv("../data/Data/NVDA.csv")

3. Filter News for Selected Stocks

In [9]:
selected_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']

news_df = news_df[news_df['stock'].isin(selected_stocks)]

news_df.head()

,Unnamed: 0,headline,url,publisher,date,stock
6680,7120,Tech Stocks And FAANGS Strong Again To Start D...,https://www.benzinga.com/government/20/06/1622...,JJ Kinahan,2020-06-10 11:33:26-04:00,AAPL
6681,7121,10 Biggest Price Target Changes For Wednesday,https://www.benzinga.com/analyst-ratings/price...,Lisa Levin,2020-06-10 08:14:08-04:00,AAPL
6682,7122,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",https://www.benzinga.com/short-sellers/20/06/1...,Benzinga Newsdesk,2020-06-10 07:53:47-04:00,AAPL
6683,7123,"Deutsche Bank Maintains Buy on Apple, Raises P...",https://www.benzinga.com/news/20/06/16219873/d...,Benzinga Newsdesk,2020-06-10 07:19:25-04:00,AAPL
6684,7124,Apple To Let Users Trade In Their Mac Computer...,https://www.benzinga.com/news/20/06/16218697/a...,Neer Varshney,2020-06-10 06:27:11-04:00,AAPL


4. Data Cleaning and Date Alignment

# Convert Dates to Datetime

In [11]:
# Convert news publication date
news_df['date'] = pd.to_datetime(
    news_df['date'],
    format='mixed',
    utc=True,
    errors='coerce'
)

# Extract only the date
news_df['date_only'] = news_df['date'].dt.date

In [12]:
# Convert stock dates
stock_datasets = {
    'AAPL': aapl_df,
    'AMZN': amzn_df,
    'GOOG': goog_df,
    'META': meta_df,
    'NVDA': nvda_df
}

for stock, df in stock_datasets.items():
    df['Date'] = pd.to_datetime(df['Date'])

5. Handle Weekend and Holiday News

Financial markets are closed on weekends and holidays.
News published during non-trading days should be aligned to the next trading day.

In [13]:
def align_to_trading_day(date):
    # If Saturday -> Monday
    if date.weekday() == 5:
        return date + BDay(1)

    # If Sunday -> Monday
    elif date.weekday() == 6:
        return date + BDay(1)

    return date

news_df['aligned_date'] = pd.to_datetime(news_df['date_only'])
news_df['aligned_date'] = news_df['aligned_date'].apply(align_to_trading_day)